# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

# Print a summary
print(f"{metadata['name']}: {metadata['description']}")
print("Published on:", metadata['datePublished'])
print("Keywords:", metadata['keywords'])
print("License:", metadata['license'])

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their @id's

record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', 'Unnamed')}")

# For each record set, list fields and columns referenced by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} - {rs.get('name', 'Unnamed')}")
    if 'field' in rs:
        print("Fields:")
        for fld in rs['field']:
            print(f"  - {fld['@id']} ({fld.get('name','')})")
    if 'column' in rs:
        print("Columns:")
        for col in rs['column']:
            print(f"  - {col['@id']} ({col.get('name','')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Identify record sets by @id

# Collect record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for Record Set {rs_id} with columns:\n{df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select a record set and numeric field by @id

# You may need to inspect the columns printed previously and update @id accordingly, e.g.:
# Let's suppose main survey data is in record set '@id': 'https://api.app.sen.science/frontiers/7160411/e1f3c048-91dc-444a-b31e-e649e1eb302f',
# and the GAD-7 score column is '@id': 'gad7_score',
# and group by 'village'

# Update these variables as per the overview above
main_record_set_id = 'https://api.app.sen.science/frontiers/7160411/e1f3c048-91dc-444a-b31e-e649e1eb302f'
numeric_field_id = 'gad7_score'  # Replace with exact column name or @id
group_field_id = 'village'  # Replace as needed

# Proceed if present
df = dataframes.get(main_record_set_id)
if df is not None and numeric_field_id in df.columns:
    # Outlier threshold (example: score > 10)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another attribute
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print(f"Could not perform EDA. DataFrame or {numeric_field_id} not found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of GAD-7 scores if column is present
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in Main Record Set")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group (e.g., village)
    if group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print(f"Visualization skipped. {numeric_field_id} column not found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded metadata and explored available record sets and fields using `mlcroissant`.
- Extracted survey records and reviewed key demographic and mental health scores (e.g., GAD-7).
- Applied basic filtering, normalization, and grouping for exploratory data analysis.
- Visualized data distributions, such as GAD-7 score histograms and boxplots by village.

**Next steps:**
- Further refine field selection for deeper analysis.
- Investigate additional field correlations and apply more domain-specific transformations.
- Use the Croissant schema via `@id` references for reproducibility and standards-compliant workflows.